# Smart-glasses candidate sector and keyword extraction
## Purpose and tutorial alignment
This notebook implements a CPC-first workflow: (1) collect candidate patent families using the current CPC list, (2) enforce the family/date/patent-type rules, (3) inspect English text coverage, and (4) extract descriptive words and phrases with TF-IDF, globally and per CPC prefix.

Collection follows Tutorial 0 `01_patstat_basics`; family counting, dates and missing-data handling follow `02_step1_general_description` and Tutorial 1 `10_patent_attributes`. **TF-IDF is our project extension**, not an implemented keyword-extraction exercise from those notebooks. Applicant harmonisation/competitive analysis from `11_competitive_landscape` is a later stage and is not performed here.

No Amplifon filter is applied. No keywords or phrase rules are used to admit or reject candidate families. Candidate membership means that at least one valid PI application carries a listed CPC prefix. It does not establish that every candidate is a smart-glasses invention. Official CPC descriptions, known-patent coverage and relevance evaluation are separate validation tasks.

One DOCDB family is one record. The family year is the minimum valid earliest filing year across its PI members. English representative text is chosen across all valid PI members: prefer an English abstract, then an English title, then earliest application filing date and smallest application ID. Families with no English text remain in sector counts and are excluded only from TF-IDF.

**Output:** query, raw family snapshot and hash, code snapshots, settings/package versions, quality tables, representative family texts, TF-IDF vocabulary/IDF and keyword rankings. Fixed data and package versions are required to reproduce scores. A deterministic cap is a development safeguard; capped data must not be described as the complete sector.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from datetime import datetime, timezone
from importlib.metadata import version

PROJECT_DIR = Path.cwd() if Path("sector_keywords.py").exists() else Path.cwd() / "project"
if not (PROJECT_DIR / "sector_keywords.py").exists():
    raise RuntimeError("Open from the project directory or repository root")
sys.path.insert(0, str(PROJECT_DIR.resolve()))

import pandas as pd
from epo.tipdata.patstat import PatstatClient
from sector_keywords import sector_query, extract_keywords, PATENT_STOPWORDS

%run "$PROJECT_DIR/key_words.ipynb"

ENV = "TEST"
DATASET_RELEASE = ""  # Required: PATSTAT release shown in TIP.
YEAR_MIN, YEAR_MAX = 2000, 2022
MAX_FAMILIES = 5000  # Deterministic cap. Increase only after checking dataset size.
SEED = 42
MIN_DF, MAX_DF, TOP_N = 2, 1.0, 30
RAW_SNAPSHOT = None  # Optional Path to a saved families_raw.json; skips database access.


## 1. Collect or replay the candidate family snapshot
The CPC list uses OR across string-prefix matches, as in the current extraction. Shared prefixes do not duplicate families. A prefix match is not a check against the official CPC hierarchy. Invalid/dummy applications, missing family IDs and non-PI rights are excluded. Invalid priority years cannot enter the selected window. The last complete analysis year must be justified for your release; this notebook does not infer completeness.

In [ ]:
if not DATASET_RELEASE.strip():
    raise ValueError("Set DATASET_RELEASE before running")
query = sector_query(CPC_CODES, YEAR_MIN, YEAR_MAX, MAX_FAMILIES, SEED)
if RAW_SNAPSHOT is None:
    patstat = PatstatClient(env=ENV)
    raw_rows = [dict(row) for row in patstat.sql_query(query, use_legacy_sql=False)]
else:
    raw_rows = json.loads(Path(RAW_SNAPSHOT).read_text(encoding="utf-8"))
    previous = json.loads((Path(RAW_SNAPSHOT).parent / "config.json").read_text())
    expected = dict(env=ENV, dataset_release=DATASET_RELEASE, codes=sorted(set(CPC_CODES)),
                    year_min=YEAR_MIN, year_max=YEAR_MAX, max_families=MAX_FAMILIES, seed=SEED)
    for key, value in expected.items():
        if previous[key] != value:
            raise ValueError(f"Replay configuration mismatch: {key}")
    raw_text = Path(RAW_SNAPSHOT).read_text(encoding="utf-8")
    if hashlib.sha256(raw_text.encode()).hexdigest() != previous["raw_sha256"]:
        raise ValueError("Raw snapshot hash mismatch")

if not raw_rows:
    raise RuntimeError("No candidates: check environment, dates and CPC selection; no relevance conclusion is possible")
families = pd.DataFrame(raw_rows)
assert not families["docdb_family_id"].duplicated().any()
assert families["priority_year"].between(YEAR_MIN, YEAR_MAX).all()
assert families["docdb_family_id"].gt(0).all()
for column in ("title", "abstract"):
    families[column] = families[column].fillna("").astype(str)

output_path = PROJECT_DIR / "outputs" / "sector_keywords" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_path.mkdir(parents=True, exist_ok=False)
raw_text = json.dumps(raw_rows, ensure_ascii=False, indent=2, default=str)
(output_path / "families_raw.json").write_text(raw_text, encoding="utf-8")
(output_path / "sector_query.sql").write_text(query, encoding="utf-8")
config = dict(env=ENV, dataset_release=DATASET_RELEASE, codes=sorted(set(CPC_CODES)),
              year_min=YEAR_MIN, year_max=YEAR_MAX, max_families=MAX_FAMILIES, seed=SEED,
              min_df=MIN_DF, max_df=MAX_DF, top_n=TOP_N, ngram_range=[1, 3],
              stopwords=PATENT_STOPWORDS, sublinear_tf=True, norm="l2",
              python=platform.python_version(),
              packages={name: version(name) for name in ("pandas", "numpy", "scipy", "scikit-learn")},
              raw_sha256=hashlib.sha256(raw_text.encode()).hexdigest(),
              replay_source=str(RAW_SNAPSHOT) if RAW_SNAPSHOT else None)
(output_path / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
for name in ("sector_keywords.py",):
    (output_path / name).write_text((PROJECT_DIR / name).read_text(), encoding="utf-8")


## 2. Check counting units, dates and language coverage
A family without English text is not automatically irrelevant. Prefix groups overlap. Coverage is reported by prefix and year to make language/time selection effects visible. These are diagnostics, not an assessment of the truth of CPC relevance.

In [ ]:
families["has_english_text"] = (families["title"] + " " + families["abstract"]).str.strip().ne("")
families["has_english_abstract"] = families["abstract"].str.strip().ne("")
total_eligible = int(families["total_eligible_families"].iloc[0])
quality = pd.DataFrame([dict(
    eligible_sector_families=total_eligible, retrieved_families=len(families),
    capped=len(families) < total_eligible,
    families_with_english_text=int(families["has_english_text"].sum()),
    english_text_coverage=float(families["has_english_text"].mean()),
    english_abstract_coverage=float(families["has_english_abstract"].mean()),
)])
by_year = families.groupby("priority_year").agg(
    families=("docdb_family_id", "size"), english_text_coverage=("has_english_text", "mean"))
membership = families.assign(cpc_prefix=families["matched_prefixes"].str.split(" | ", regex=False)).explode("cpc_prefix")
by_prefix = membership.groupby("cpc_prefix").agg(
    sampled_families=("docdb_family_id", "nunique"),
    english_text_coverage=("has_english_text", "mean"))
quality.to_csv(output_path / "quality_summary.csv", index=False)
by_year.to_csv(output_path / "quality_by_year.csv")
by_prefix.to_csv(output_path / "quality_by_prefix.csv")
display(quality, by_year, by_prefix)
if len(families) < total_eligible:
    print("CAPPED SAMPLE: keywords and year counts describe the retrieved sample, not the full sector.")


## 3. Extract descriptive keywords and phrases
Fit TF-IDF once on all readable retrieved families. Reuse the same vocabulary and IDF for each CPC group. Rank phrases by mean L2-normalised TF-IDF and also report family frequency/prevalence and all supporting family IDs. Scores describe vocabulary, not relevance or Amplifon-specific opportunities. A per-CPC high score is not a statistical test of distinctiveness.

Processing: lowercase, Unicode accent stripping, alphabetic tokens of at least two characters, explicit versioned stopwords, 1–3 token n-grams and sublinear TF. There is no lemmatisation. N-grams are formed after stopword removal and can therefore join words that were separated by stopwords. No terms are silently added to the extraction query.

In [ ]:
keywords, vectorizer, readable, status = extract_keywords(
    families, min_df=MIN_DF, max_df=MAX_DF, top_n=TOP_N,
)
keywords.to_csv(output_path / "keywords.csv", index=False)
if vectorizer is not None:
    pd.DataFrame({"phrase": vectorizer.get_feature_names_out(), "idf": vectorizer.idf_}).to_csv(
        output_path / "vocabulary_idf.csv", index=False)
display(keywords[keywords["scope"] == "ALL_SECTOR"])
print(f"TF-IDF status: {status}")
# Inspect this table to compare the descriptive vocabulary of each prefix.
display(keywords[keywords["scope"] != "ALL_SECTOR"])


In [ ]:
# Save representative family texts to trace keywords back to their source.
families.to_csv(output_path / "family_evidence.csv", index=False)
print(f"Saved snapshot, quality checks and TF-IDF results to {output_path.resolve()}")